<a href="https://colab.research.google.com/github/ChyTalksTech/Inventory_Write-offs/blob/feature%2Fdata-ingestion/Inventory_Write_offs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimising the Health of Global Inventory through Automated Write-off prediction and root cause clustering

### Github Repository Link:

### AI Assistance Log

library Imports: Pandas, Sklearn, XGBoost, TensorFlow, SHAP

In [11]:
import pandas as pd


## #1 Data Ingestion and Cleaning

initial cleaning and joins were done in snowflake via a query


### Pre-processing & SQL Extraction

SELECT
    m.HVR_ROWID AS "ROW_ID",
    m.MJAHR AS "YEAR",
    m.BUDAT_MKPF As "DATE",
    m.zz_gbrand AS BRAND_ID,
    z.zbrand AS "BRAND_NAME",
    m.MATNR AS "MSEG_MATERIAL_NUMBER",
    ma.MAKTX as "MATERIAL_NAME",
    a.MATERIAL_TYPE_CODE AS "MATERIAL_TYPE_CODE",
    a.MATERIAL_TYPE_DESCRIPTION AS "MATERIAL_TYPE_DESCRIPTION",
    m.BWART AS "MOVEMENT_TYPE_CODE",
    t1.BTEXT as "MOVEMENT_TYPE",
    m.SAKTO AS "G/L_ACCOUNT",
    s.TXT50 AS "G/L_ACCOUNT_NAME",
    m.WAERS AS "lOCAL_CURRENCY",
    b.UKURS AS "CURRENCY_CONVERSION",
    m.dmbtr AS "AMOUNT_IN_lOCAL_CURRENCY",
    m.dmbtr * ABS(b.ukurs) AS "AMOUNT_IN_USD",
    m.bualt AS "ALTERNATIVE_AMOUNT",
    m.bwtar AS "VALUATION_TYPE",
    m.menge * q.scc_usd AS "GROUP_Cost_USD",
FROM "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."MSEG" AS m
LEFT JOIN "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."T001" AS cc
ON m.BUKRS = cc.BUKRS AND cc.SPRAS = 'E'
LEFT JOIN "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."SKAT" AS s
ON m.SAKTO = s.SAKNR AND s.SPRAS = 'E' AND cc.KTOPL = s.KTOPL
LEFT JOIN "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."T001W" AS t0
ON m.WERKS = t0.WERKS AND t0.SPRAS = 'E'
LEFT JOIN "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."T156T" AS t1
ON m.BWART = t1.BWART AND m.SOBKZ = t1.SOBKZ AND t1.SPRAS = 'E' AND  t1.kzvbr =''
LEFT JOIN "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."MAKT" AS ma
ON m.MATNR = ma.MATNR AND ma.SPRAS = 'E'
LEFT JOIN "ENTDATA_PUB"."ODH_SAP_MDG_SS"."ZMBRAND_MDG" AS z
ON m.zz_gbrand = z.zzbrandid
LEFT JOIN(
    SELECT ma2.MATNR, ma2.MTART AS MATERIAL_TYPE_CODE, t3.MTBEZ as MATERIAL_TYPE_DESCRIPTION
    FROM "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."MARA" as ma2
    LEFT JOIN "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."T134T" AS t3
    ON ma2.MTART = t3.MTART AND t3.SPRAS = 'E'
    ) AS a
ON m.matnr = a.matnr
LEFT JOIN(
    SELECT
        t.FCURR,
        t.TCURR,
        t.UKURS,
        t.ZZLTDATE
    FROM "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."TCURR" t
    INNER JOIN (
        SELECT
            FCURR,
            MAX(ZZLTDATE) AS MaxDate
        FROM "ENTDATA_PUB"."ODH_SAP_ECC_NA_SS"."TCURR"
        WHERE TCURR = 'USD' AND KURST = 'M'
        GROUP BY FCURR
    ) x
    ON  t.FCURR = x.FCURR
    AND t.ZZLTDATE = x.MaxDate
    WHERE t.TCURR = 'USD' AND KURST = 'M'
     ) as b
ON CASE
    WHEN m.WAERS = 'USD' THEN 'USDN'
    ELSE m.WAERS
    END = b.FCURR
LEFT JOIN ENTDATA_PUB.ODH_SAP_MDG_SS.MARC as i
ON m.werks = i.werks AND m.matnr = i.matnr
LEFT JOIN OPS_DP_PUB.INVENTORY.GROUP_COST_SCC as q
ON q.PRODUCT_CODE = m.MATNR AND m.werks = q.plant_code AND q.organization_code = i.zorgid
WHERE s.saknr IN ('0040640100', '0044890100', '0040650120', '0040650140', '0040650100') AND m.MJAHR = 2025 AND a.MATERIAL_TYPE_DESCRIPTION = 'Finished Material';

In [12]:
MSEG_df = pd.read_csv('MSEG File.csv')

MSEG_df.head()

,ROW_ID,YEAR,DATE,BRAND_ID,BRAND_NAME,MSEG_MATERIAL_NUMBER,MATERIAL_NAME,MATERIAL_TYPE_CODE,MATERIAL_TYPE_DESCRIPTION,MOVEMENT_TYPE_CODE,MOVEMENT_TYPE,G/L_ACCOUNT,G/L_ACCOUNT_NAME,lOCAL_CURRENCY,CURRENCY_CONVERSION,AMOUNT_IN_lOCAL_CURRENCY,AMOUNT_IN_USD,ALTERNATIVE_AMOUNT,VALUATION_TYPE,GROUP_Cost_USD
0,20742465,2025,20250516,1003,CRESTOR,310075231,CRESTOR TAB 20MG BT 1X30 EA KR,ZFIN,Finished Material,988,Misc. Inv Adj - BLK,44890100,Inventory Write-off - Scrap / Destruction,USD,-1.0,0.0,0.0,8.29,NaN,NaN
1,20740096,2025,20250515,1017,ZESTRIL,310914512,ZESTORETIC TAB 20/25MG BT 1X100 EA CA,ZFIN,Finished Material,988,Misc. Inv Adj - BLK,44890100,Inventory Write-off - Scrap / Destruction,USD,-1.0,0.0,0.0,110.73,NaN,NaN
2,20740083,2025,20250515,1017,ZESTRIL,310914212,ZESTORETIC TAB 20/12.5MG BT 1X100 EA CA,ZFIN,Finished Material,988,Misc. Inv Adj - BLK,44890100,Inventory Write-off - Scrap / Destruction,USD,-1.0,0.0,0.0,60.28,NaN,NaN
3,20736800,2025,20250515,1023,TOPROL AG,00310930305,METOPROLOL SUCCINATE 200MG 100CT LANNETT,ZFIN,Finished Material,988,Misc. Inv Adj - BLK,44890100,Inventory Write-off - Scrap / Destruction,USD,-1.0,0.0,0.0,304.54,NaN,NaN
4,20736827,2025,20250515,1070,SELOKEN ZOK,00310709005,TOPROL XL 50MG 100CT ARALEZ,ZFIN,Finished Material,988,Misc. Inv Adj - BLK,44890100,Inventory Write-off - Scrap / Destruction,USD,-1.0,0.0,0.0,91.03,NaN,NaN


In [13]:
MSEG_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1494 entries, 0 to 1493
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ROW_ID                     1494 non-null   int64  
 1   YEAR                       1494 non-null   int64  
 2   DATE                       1494 non-null   int64  
 3   BRAND_ID                   1494 non-null   int64  
 4   BRAND_NAME                 1494 non-null   object 
 5   MSEG_MATERIAL_NUMBER       1494 non-null   object 
 6   MATERIAL_NAME              1494 non-null   object 
 7   MATERIAL_TYPE_CODE         1494 non-null   object 
 8   MATERIAL_TYPE_DESCRIPTION  1494 non-null   object 
 9   MOVEMENT_TYPE_CODE         1494 non-null   int64  
 10  MOVEMENT_TYPE              1494 non-null   object 
 11  G/L_ACCOUNT                1494 non-null   int64  
 12  G/L_ACCOUNT_NAME           1494 non-null   object 
 13  lOCAL_CURRENCY             1494 non-null   objec

Notes:

*   Date should be date not int64
*   Valuation Type is fully null so can delete
*   Some group cost is null. can rectify by forward filling material first and then using Median if Material type does not have other dates






In [14]:
# Converting Date from Int to Date
MSEG_df['DATE'] = pd.to_datetime(MSEG_df['DATE'].astype(str), format='%Y%m%d')

# Removing Valuation Type Column
MSEG_df.drop(columns=['VALUATION_TYPE'], inplace=True)

MSEG_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1494 entries, 0 to 1493
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   ROW_ID                     1494 non-null   int64         
 1   YEAR                       1494 non-null   int64         
 2   DATE                       1494 non-null   datetime64[ns]
 3   BRAND_ID                   1494 non-null   int64         
 4   BRAND_NAME                 1494 non-null   object        
 5   MSEG_MATERIAL_NUMBER       1494 non-null   object        
 6   MATERIAL_NAME              1494 non-null   object        
 7   MATERIAL_TYPE_CODE         1494 non-null   object        
 8   MATERIAL_TYPE_DESCRIPTION  1494 non-null   object        
 9   MOVEMENT_TYPE_CODE         1494 non-null   int64         
 10  MOVEMENT_TYPE              1494 non-null   object        
 11  G/L_ACCOUNT                1494 non-null   int64         
 12  G/L_AC

In [15]:
#1. Forward filling in nulls in Group Cost


# Sort by material and date first
MSEG_df = MSEG_df.sort_values(by=['MSEG_MATERIAL_NUMBER', 'DATE'])

# Forward fill costs within each material group
MSEG_df['GROUP_Cost_USD'] = MSEG_df.groupby('MSEG_MATERIAL_NUMBER')['GROUP_Cost_USD'].ffill()

MSEG_df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 1494 entries, 14 to 1194
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   ROW_ID                     1494 non-null   int64         
 1   YEAR                       1494 non-null   int64         
 2   DATE                       1494 non-null   datetime64[ns]
 3   BRAND_ID                   1494 non-null   int64         
 4   BRAND_NAME                 1494 non-null   object        
 5   MSEG_MATERIAL_NUMBER       1494 non-null   object        
 6   MATERIAL_NAME              1494 non-null   object        
 7   MATERIAL_TYPE_CODE         1494 non-null   object        
 8   MATERIAL_TYPE_DESCRIPTION  1494 non-null   object        
 9   MOVEMENT_TYPE_CODE         1494 non-null   int64         
 10  MOVEMENT_TYPE              1494 non-null   object        
 11  G/L_ACCOUNT                1494 non-null   int64         
 12  G/L_ACCOUN

In [16]:
# Still some nulls so will use median

# Calculate median cost per Material Type (e.g., 'Finished Material')
material_type_medians = MSEG_df.groupby('MATERIAL_TYPE_DESCRIPTION')['GROUP_Cost_USD'].transform('median')

# Fill remaining nulls with the group median
MSEG_df['GROUP_Cost_USD'] = MSEG_df['GROUP_Cost_USD'].fillna(material_type_medians)

MSEG_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 1494 entries, 14 to 1194
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   ROW_ID                     1494 non-null   int64         
 1   YEAR                       1494 non-null   int64         
 2   DATE                       1494 non-null   datetime64[ns]
 3   BRAND_ID                   1494 non-null   int64         
 4   BRAND_NAME                 1494 non-null   object        
 5   MSEG_MATERIAL_NUMBER       1494 non-null   object        
 6   MATERIAL_NAME              1494 non-null   object        
 7   MATERIAL_TYPE_CODE         1494 non-null   object        
 8   MATERIAL_TYPE_DESCRIPTION  1494 non-null   object        
 9   MOVEMENT_TYPE_CODE         1494 non-null   int64         
 10  MOVEMENT_TYPE              1494 non-null   object        
 11  G/L_ACCOUNT                1494 non-null   int64         
 12  G/L_ACCOUN


**Data Rectification & Financial Integrity Strategy**
Problem: Initial data profiling revealed missing values in the GROUP_COST field (approx. 28% of records). These nulls typically occur due to synchronization lags between the SAP MSEG transaction tables and the SCC cost master tables.

Implementation Choice: A Hierarchical Imputation strategy was selected over simple deletion to preserve the financial materiality of the dataset:

Step 1: Temporal Forward-Fill: Leverages stability of pharmaceutical pricing by carrying over the last known cost for a specific SKU. This filled in about 40 records.

Step 2: Categorical Median Imputation: For new products without historical costs, the median value of the specific MATERIAL_TYPE_DESCRIPTION (e.g., Finished Material) is applied.

Justification: This dual-layered approach minimizes attribution bias and ensures that the Value-at-Risk is not under-reported in the subsequent modeling phases.

In [17]:
# Metadata Decoupling

#Preserving Names for final report
material_lookup = MSEG_df[['MSEG_MATERIAL_NUMBER', 'MATERIAL_NAME', 'MATERIAL_TYPE_CODE', 'MATERIAL_TYPE_DESCRIPTION']].drop_duplicates().set_index('MSEG_MATERIAL_NUMBER')
brand_lookup = MSEG_df[['BRAND_ID', 'BRAND_NAME']].drop_duplicates().set_index('BRAND_ID')
movement_lookup = MSEG_df[['MOVEMENT_TYPE_CODE', 'MOVEMENT_TYPE']].drop_duplicates().set_index('MOVEMENT_TYPE_CODE')
account = MSEG_df[['G/L_ACCOUNT', 'G/L_ACCOUNT_NAME']].drop_duplicates().set_index('G/L_ACCOUNT')


In [18]:
# Drop text descriptions to save memory and prevent multi-collinearity
MSEG_df = MSEG_df.drop(columns=[
    'MATERIAL_NAME',
    'BRAND_NAME',
    'MOVEMENT_TYPE',
    'MATERIAL_TYPE_DESCRIPTION',
    'G/L_ACCOUNT_NAME',
    'lOCAL_CURRENCY',
    'CURRENCY_CONVERSION',
    'AMOUNT_IN_lOCAL_CURRENCY',
    'ALTERNATIVE_AMOUNT',
    'YEAR',
    'ROW_ID',
])

print(f"Fact Table pruned. {len(MSEG_df.columns)} features remaining for modeling.")

Fact Table pruned. 8 features remaining for modeling.


**Relational Decoupling and Feature Pruning**
Objective: To optimize the dataset for high-dimensional modeling (LSTM and Anomaly Detection) while maintaining downstream interpretability.

Process:

Dimension Extraction: Text-based descriptors (Material Names, Brand Names) were extracted into separate lookup dictionaries.

Redundancy Reduction: The main dataframe was pruned to keep only the numeric "Primary Keys" and "Fact" values (Costs, Quantities).

Justification: This prevents the model from attempting to find patterns in high-cardinality string data, which reduces noise and memory usage. The decoupled metadata is preserved to allow the final "High-Risk" SKUs to be mapped back to human-readable names for executive reporting.

In [19]:
MSEG_df.head()

,DATE,BRAND_ID,MSEG_MATERIAL_NUMBER,MATERIAL_TYPE_CODE,MOVEMENT_TYPE_CODE,G/L_ACCOUNT,AMOUNT_IN_USD,GROUP_Cost_USD
14,2025-12-19,1511,00310027110,ZFIN,988,44890100,0.0,78.672694
108,2025-12-19,1511,00310027110,ZFIN,988,44890100,0.0,78.672694
1433,2025-12-19,1511,00310027210,ZFIN,988,44890100,0.0,78.672694
1380,2025-12-19,1511,00310027460,ZFIN,988,44890100,0.0,78.672694
1387,2025-12-19,1511,00310027460,ZFIN,988,44890100,0.0,78.672694
